# Prophet 개요
- 메타에서 개발한 시계열 예측 라이브러리

# 1. 준비작업

In [1]:
from hossam import *

from matplotlib import pyplot as plt
import seaborn as sb
import numpy as np
from pandas import to_datetime, DataFrame
from prophet import Prophet
from prophet.plot import add_changepoints_to_plot
from sklearn.metrics import mean_absolute_error, mean_squared_error

📦 아이티윌 이광호 강사가 제작한 라이브러리를 사용중입니다.
📚 자세한 사용 방법은 https://py.hossam.kr 을 참고하세요.
📧 Email: leekh4232@gmail.com
🎬 Youtube: https://www.youtube.com/@hossam-codingclub
📝 Blog: https://blog.hossam.kr/
🔖 Version: 0.5.3

✅ 시각화를 위한 한글 글꼴(NotoSansKR-Regular)이 자동 적용되었습니다.


### 1. 데이터 가져오기

In [2]:
origin = load_data('air_passengers')
print(f"데이터셋 크기: {origin.shape}")
print(f"열 개수: {origin.shape[1]}")
print(f"행 개수: {origin.shape[0]}")
print(origin.info())
origin.head()

어느 항공사의 월간 탑승객 수 (출처: https://www.kaggle.com/datasets/rakannimer/air-passengers)
데이터셋 크기: (144, 2)
열 개수: 2
행 개수: 144
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 144 entries, 0 to 143
Data columns (total 2 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   Month       144 non-null    datetime64[ns]
 1   Passengers  144 non-null    int64         
dtypes: datetime64[ns](1), int64(1)
memory usage: 2.4 KB
None


,Month,Passengers
0,1949-01-01,112
1,1949-02-01,118
2,1949-03-01,132
3,1949-04-01,129
4,1949-05-01,121


## 2. 시계열 데이터 준비
### 1. 데이터 분할
#### 1. 단순 비율 분할(ex. 80/20)

In [3]:
df_prophet = origin.rename(columns = {'Month': 'ds', 'Passengers': 'y'})
df_prophet['ds'] = to_datetime(df_prophet['ds'])
df_prophet.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 144 entries, 0 to 143
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   ds      144 non-null    datetime64[ns]
 1   y       144 non-null    int64         
dtypes: datetime64[ns](1), int64(1)
memory usage: 2.4 KB


In [5]:
# 분할 비율
split_ratio = 0.8

# 분할 인덱스
split_idx = int(len(df_prophet) * split_ratio)

# 훈련 / 검증 데이터
train = df_prophet.iloc[:split_idx]
test = df_prophet.iloc[split_idx:]

print('Train 기간:', train['ds'].min(), '~', train['ds'].max())
print('Valid 기간:', test['ds'].min(), '~', test['ds'].max())

Train 기간: 1949-01-01 00:00:00 ~ 1958-07-01 00:00:00
Valid 기간: 1958-08-01 00:00:00 ~ 1960-12-01 00:00:00


#### 2. 날짜 기준 분할 (예: 특정 시점 이후를 검증)

In [6]:
# 검증 시작일 설정
split_date = '1958-08-01'

train = df_prophet[df_prophet['ds'] < split_date]
test = df_prophet[df_prophet['ds'] >= split_date]


print('Train 기간:', train['ds'].min(), '~', train['ds'].max())
print('Valid 기간:', test['ds'].min(), '~', test['ds'].max())

Train 기간: 1949-01-01 00:00:00 ~ 1958-07-01 00:00:00
Valid 기간: 1958-08-01 00:00:00 ~ 1960-12-01 00:00:00


### 2. 공휴일 데이터 설정
#### 1. Air Passengers Dataset의 특성
- IF 공휴일 기대효과가 없다? 설정 안해줘도 됨

#### 2. 공휴일 데이터 생성하기
- 컬럼명이 고정되어 있음


In [7]:
# 전체 연도 범위
years = range(1949, 1961)

holiday_list = []

for y in years:
    holiday_list.extend([
        # Independence Day -> 7월 효과
        {'holiday': 'independence_day', 'ds': f'{y}-07-01'},
        # Thanksgiving Day -> 11월 효과
        {'holiday': 'thanksgiving_day', 'ds': f'{y}-11-01'},
        # Christmas -> 12월 효과
        {'holiday': 'christmas', 'ds': f'{y}-12-01'},
        # Summer Vacation 시작 -> 6월 효과
        {'holiday': 'summer_vacation', 'ds': f'{y}-06-01'},
    ])

holidays = DataFrame(holiday_list)
holidays['ds'] = to_datetime(holidays['ds'])

# 공휴일 효과 없음으로 설정
holidays['lower_window'] = 0
holidays['upper_window'] = 0

holidays.head()

,holiday,ds,lower_window,upper_window
0,independence_day,1949-07-01,0,0
1,thanksgiving_day,1949-11-01,0,0
2,christmas,1949-12-01,0,0
3,summer_vacation,1949-06-01,0,0
4,independence_day,1950-07-01,0,0


#### 3. 공휴일 효과 포함 샘플 코드


In [8]:
hd_test = DataFrame({
    'holiday': 'christmas',
    'ds': to_datetime(['2020-12-25']),
    'lower_window': -2,
    'upper_window': 1
})

hd_test

,holiday,ds,lower_window,upper_window
0,christmas,2020-12-25,-2,1


## 3. 시계열 학습 모델 구현
### 1. 기본 모형 설정
#### 1. 데이터 유형별 권장 조합

In [12]:
model = Prophet(
    growth = 'linear',
    changepoint_prior_scale = 0.05,
    seasonality_mode = 'additive',
    yearly_seasonality = True,
    weekly_seasonality = False,
    daily_seasonality = False,
    holidays = holidays,
    seasonality_prior_scale = 10,
    holidays_prior_scale = 10
)

model.fit(train)

AttributeError: 'Prophet' object has no attribute 'stan_backend'

In [ ]:
import prophet
print("prophet module:", prophet.__file__)``
print("prophet version:", prophet.__version__)
from prophet import Prophet
print("Prophet class:", Prophet)
``

prophet module: c:\Users\wodyd\AppData\Local\Programs\Python\Python311\Lib\site-packages\prophet\__init__.py
prophet version: 1.3.0
Prophet class: <class 'prophet.forecaster.Prophet'>
